In [22]:
from qiskit import QuantumCircuit
from qiskit_aer import AerSimulator
from qiskit_aer.noise import NoiseModel, pauli_error

In [23]:
# Noise: each qubit flips with probability p
p = 0.1

bitflip = pauli_error([('X', p), ('I', 1 - p)])

noise = NoiseModel()
noise.add_all_qubit_quantum_error(bitflip, ['id'])

sim = AerSimulator(noise_model=noise)

shots = 10000

In [24]:
# Without error correction (1 qubit)
plain = QuantumCircuit(1, 1)

plain.x(0)       # Logical value 1
plain.id(0)      # Noisy step
plain.measure(0, 0)

In [25]:
# With error correction (3-qubit repetition code)
protected = QuantumCircuit(3, 1)

protected.x(0)

# Encoding: |1> -> |111>
protected.cx(0, 1)
protected.cx(0, 2)

protected.barrier()

# Apply noise
protected.id([0, 1, 2])

protected.barrier()

CircuitInstruction(operation=Instruction(name='barrier', num_qubits=3, num_clbits=0, params=[]), qubits=(<Qubit register=(3, "q"), index=0>, <Qubit register=(3, "q"), index=1>, <Qubit register=(3, "q"), index=2>), clbits=())

In [26]:
# Decode and detect errors
protected.cx(0, 1)
protected.cx(0, 2)

# Correct q0 using majority voting
protected.ccx(1, 2, 0)

# Measure corrected qubit
protected.measure(0, 0)

In [ ]:
def error_rate(circuit):
    counts = sim.run(circuit, shots=shots).result().get_counts()

    return counts.get('0', 0) / shots